In [1]:
# Initialize Neo4j connectionfrom pydantic import BaseModel, Field
from typing import List, Optional
from datetime import date
from enum import Enum
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from pydantic import BaseModel, Field


/home/vxh/RAG-advanced/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name=embeddings_model_name)
loader = TextLoader('data/doc.txt', encoding='utf-8')
single_doc = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(single_doc)

for i, doc in enumerate(documents):
    doc.metadata["source"] = f"chunk_{i}"

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedder, 
    collection_name="post_retrieval_db"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10364.01it/s]


Task 1: Define Domain Schema 

In [4]:

from pydantic import BaseModel, Field
from typing import List

class Constraint(BaseModel):
    """Giới hạn hoặc điều kiện có thể đo lường được."""
    metric: str = Field(description="Đơn vị hoặc tiêu chí đo lường (VD: 'ngày', 'VND', '%')")
    value: float = Field(description="Giá trị con số cụ thể")

class Commitment(BaseModel):
    """Quy định, nghĩa vụ hoặc cam kết cụ thể."""
    description: str = Field(description="Nội dung chi tiết của quy định")
    measurable: bool = Field(description="True nếu quy định có chứa con số giới hạn cụ thể")
    constraints: List[Constraint] = Field(default=[], description="Danh sách các giới hạn nếu measurable là True")

class PolicyExtraction(BaseModel):
    """Node cấp cao nhất đại diện cho một điều khoản chính sách."""
    policy_name: str = Field(description="Tên hoặc chủ đề chính của chính sách (VD: 'Nghỉ phép năm', 'Làm thêm giờ')")
    stakeholders: List[str] = Field(default=[], description="Đối tượng áp dụng (VD: 'Nhân viên', 'Quản lý')")
    regulations: List[str] = Field(default=[], description="Các văn bản, luật hoặc mã tài liệu liên quan")
    commitments: List[Commitment] = Field(default=[], description="Danh sách các quy định chi tiết trong chính sách này")

class DocumentExtraction(BaseModel):
    """Wrapper để bắt nhiều chính sách trong cùng một chunk text."""
    policies: List[PolicyExtraction] = Field(default=[])

print("✅ Đã khởi tạo xong Pydantic Schemas!")

✅ Đã khởi tạo xong Pydantic Schemas!


Task 2: Entity and Relationship Extraction

In [10]:
# ==========================================
# CELL 2: EXTRACTION PIPELINE
# ==========================================
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Khởi tạo model (Nhiệt độ = 0 để output mang tính xác định, không bay bổng)

llm = ChatOllama(model="llama3.1", temperature=0)
structured_extractor = llm.with_structured_output(DocumentExtraction)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """Bạn là một cỗ máy trích xuất dữ liệu Nhân sự (HR).
    Nhiệm vụ: Trích xuất TOÀN BỘ chính sách, đối tượng áp dụng, văn bản liên quan và quy định từ văn bản.
    
    QUY TẮC NGHIÊM NGẶT:
    1. KHÔNG tự bịa thông tin. Nếu không có trong text, để trống.
    2. Nếu quy định có chứa con số (vd: 'thanh toán 150%'), đặt measurable=True và trích xuất Constraint.
    3. Trích xuất Stakeholder kể cả khi nói ẩn ý (vd: 'áp dụng cho toàn bộ người lao động' -> 'Nhân viên').
    """),
    ("human", "Văn bản cần trích xuất:\n\n{chunk}")
])

extraction_chain = extraction_prompt | structured_extractor
chunks=[doc.page_content for doc in documents]

print("Đang chạy pipeline trích xuất...")
extracted_data = extraction_chain.invoke({"chunk": chunks})
print(extracted_data.model_dump_json(indent=2))

Đang chạy pipeline trích xuất...
{
  "policies": [
    {
      "policy_name": "FPT INTERNAL POLICY",
      "stakeholders": [
        "Nhân viên"
      ],
      "regulations": [
        "Làm việc 8 giờ/ngày, từ thứ 2 đến thứ 6.",
        "Giờ làm việc tiêu chuẩn: 08:30 - 12:00 (sáng), 13:30 - 17:30 (chiều).",
        "Nghỉ trưa: 12:00 - 13:30.",
        "Phải có mặt ít nhất 5 phút trước giờ làm việc.",
        "Có thể linh hoạt giờ làm việc tùy theo sự cho phép của bộ phận.",
        "Làm việc từ xa phải được cấp trên trực tiếp phê duyệt."
      ],
      "commitments": [
        {
          "description": "Phải kiểm tra vào/ra khỏi công ty qua hệ thống myFPT",
          "measurable": true,
          "constraints": [
            {
              "metric": "Giờ làm việc",
              "value": 8.0
            },
            {
              "metric": "Thời gian nghỉ trưa",
              "value": 1.5
            }
          ]
        },
        {
          "description": "Phải có mặt ít nhấ

In [11]:
graph = Neo4jGraph(url="bolt://localhost:7687", username="neo4j", password="12345678")
from langchain_neo4j import Neo4jGraph
def push_to_neo4j(extraction: DocumentExtraction):
    for policy in extraction.policies:
        # 1. Tạo node PolicyClause
        graph.query("""
            MERGE (p:PolicyClause {name: $name})
            """, {"name": policy.policy_name})

        # 2. Tạo node Stakeholders & Edge AFFECTS
        for stakeholder in policy.stakeholders:
            graph.query("""
                MERGE (s:Stakeholder {name: $s_name})
                WITH s
                MATCH (p:PolicyClause {name: $p_name})
                MERGE (p)-[:AFFECTS]->(s)
                """, {"s_name": stakeholder, "p_name": policy.policy_name})

        # 3. Tạo node Regulation & Edge REFERENCES
        for reg in policy.regulations:
            graph.query("""
                MERGE (r:Regulation {name: $r_name})
                WITH r
                MATCH (p:PolicyClause {name: $p_name})
                MERGE (p)-[:REFERENCES]->(r)
                """, {"r_name": reg, "p_name": policy.policy_name})

        # 4. Tạo node Commitment & Edge CONTAINS
        for com in policy.commitments:
            graph.query("""
                MERGE (c:Commitment {description: $desc})
                SET c.measurable = $meas
                WITH c
                MATCH (p:PolicyClause {name: $p_name})
                MERGE (p)-[:CONTAINS]->(c)
                """, {"desc": com.description, "meas": com.measurable, "p_name": policy.policy_name})

print("✅ Đã định nghĩa hàm đẩy dữ liệu lên Neo4j!")


✅ Đã định nghĩa hàm đẩy dữ liệu lên Neo4j!


Task4

In [12]:
# ==========================================
# CELL 4: GRAPHRAG QUERY PIPELINE (FIXED)
# ==========================================
from langchain_neo4j import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

# Prompt 1: Dạy LLM cách chuyển câu hỏi thành lệnh Cypher
cypher_template = """Bạn là một chuyên gia Neo4j. Hãy chuyển câu hỏi sau thành lệnh Cypher.
Chỉ sử dụng các Node và Relationship có trong schema sau đây:
{schema}

QUY TẮC TỐI THƯỢNG:
1. TRONG MỆNH ĐỀ RETURN, TUYỆT ĐỐI KHÔNG SỬ DỤNG ALIAS (BÍ DANH) CÓ DẤU CÁCH HOẶC DẤU NGOẶC KÉP. 
   - SAI: RETURN s.name AS "Đối tượng áp dụng"
   - ĐÚNG: RETURN s.name, p.name
2. Luôn dùng toLower() khi tìm kiếm text.

Ví dụ 1:
Câu hỏi: "Những chính sách nào áp dụng cho Nhân viên?"
Cypher: MATCH (p:PolicyClause)-[:AFFECTS]->(s:Stakeholder) WHERE toLower(s.name) CONTAINS 'nhân viên' RETURN p.name

Ví dụ 2:
Câu hỏi: "Chính sách nghỉ phép năm có những quy định gì?"
Cypher: MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment) WHERE toLower(p.name) CONTAINS 'nghỉ phép' RETURN p.name, c.description

Câu hỏi: {question}
Lệnh Cypher:"""

cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"], 
    template=cypher_template
)

# Prompt 2: Dạy LLM cách trả lời dựa trên kết quả database
qa_template = """Bạn là trợ lý HR của FPT. Dựa vào thông tin rút trích từ cơ sở dữ liệu dưới đây, hãy trả lời câu hỏi của người dùng.
Tuyệt đối không nhắc đến "Database", "Node", "Cypher" hay "Neo4j" trong câu trả lời của bạn. Trả lời một cách tự nhiên.

Thông tin từ hệ thống: {context}

Câu hỏi: {question}
Trả lời:"""

qa_prompt = PromptTemplate(
    input_variables=["context", "question"], 
    template=qa_template
)

# Khởi tạo Chain
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True, # Bật True để xem lệnh Cypher sinh ra ở backend, rất tốt để debug!
    cypher_prompt=cypher_prompt,
    qa_prompt=qa_prompt,
    allow_dangerous_requests=True
)

# Test Pipeline
query = "Chính sách nghỉ phép năm áp dụng cho những đối tượng nào?"
print(f"User: {query}")
try:
    response = graph_chain.invoke({"query": query})
    print(f"\nAgent: {response['result']}")
except Exception as e:
    print(f"\n[LỖI CỨNG]: {e}")

User: Chính sách nghỉ phép năm áp dụng cho những đối tượng nào?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment)-[:AFFECTS]->(s:Stakeholder)
WHERE toLower(p.title) CONTAINS 'nghỉ phép'
RETURN p.name, c.description, s.name



Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `AFFECTS` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=54, offset=53>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 53, 'line': 1, 'column': 54}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment)-[:AFFECTS]->(s:Stakeholder)\nWHERE toLower(p.title) CONTAINS 'nghỉ phép'\nRETURN p.name, c.description, s.name\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `

Full Context:
[]

> Finished chain.

Agent: Chính sách nghỉ phép năm được áp dụng cho tất cả nhân viên FPT, bao gồm cả nhân viên chính thức và nhân viên hợp đồng.


In [13]:

import time

# Danh sách 10 câu hỏi kiểm thử (Test Set)
test_queries = [
    # --- 1. Entity lookup queries (Tìm kiếm cơ bản) ---
    "Tìm cho tôi thông tin về chính sách làm thêm giờ (Overtime).",
    "Liệt kê tất cả các đối tượng (Stakeholders) được nhắc đến trong các chính sách của công ty.",
    "Liệt kê các quy định hoặc cam kết có thể đo lường được (measurable = true).",
    "Có những văn bản quy định (Regulation) nào mã là FPT-HR được tham chiếu không?",
    "Có những nội dung chính sách nào liên quan đến nghỉ phép năm (Annual Leave)?",
    
    # --- 2. Relationship traversal queries (Duyệt quan hệ đa lúp) ---
    "Những chính sách nào được áp dụng cho Nhân viên (Employee)?",
    "Chính sách làm thêm giờ (Overtime) bao gồm những quy định chi tiết nào?",
    "Cho tôi biết chính sách Nghỉ phép năm (Annual Leave) tham chiếu đến văn bản nào và áp dụng cho ai?",
    
    # --- 3. Aggregation queries (Tích hợp, đếm, gom nhóm) ---
    "Mỗi chính sách hiện hành có bao nhiêu quy định (Commitments) đi kèm?",
    "Hãy liệt kê tên các chính sách và danh sách gộp tất cả các đối tượng (Stakeholders) chịu ảnh hưởng bởi từng chính sách đó."
]

print("🚀 BẮT ĐẦU CHẠY AUTOMATED TEST CHO GRAPHRAG...\n")
print("=" * 80)

# Khởi tạo biến đếm số lượng pass/fail
passed = 0
failed = 0

for i, query in enumerate(test_queries, 1):
    print(f"\n📝 TEST CASE {i}/10:")
    print(f"🧑 User: {query}")
    print("-" * 40)
    print("🔄 Agent đang suy nghĩ và dịch sang Cypher...")
    
    start_time = time.time()
    
    try:
        response = graph_chain.invoke({"query": query})
        execution_time = time.time() - start_time
        print(f"\n✨ Agent ({execution_time:.2f}s): {response['result']}")
        passed += 1
        
    except Exception as e:
        print(f"\n❌ [LỖI TRUY VẤN CYPHER]: {e}")
        failed += 1
        
    print("=" * 80)

# In tổng kết
print("\n📊 TỔNG KẾT BÀI TEST:")
print(f"✅ Thành công: {passed}/10")
print(f"❌ Thất bại: {failed}/10")

if failed == 0:
    print("🎉 Tuyệt vời! Prompt Cypher của bạn đã hoạt động hoàn hảo với mọi loại truy vấn.")
else:
    print("⚠️ Có câu hỏi bị lỗi. Hãy xem lại câu lệnh Cypher sinh ra ở các câu thất bại và bổ sung 'Ví dụ' vào cypher_template ở Cell 4 để dạy lại LLM.")

🚀 BẮT ĐẦU CHẠY AUTOMATED TEST CHO GRAPHRAG...


📝 TEST CASE 1/10:
🧑 User: Tìm cho tôi thông tin về chính sách làm thêm giờ (Overtime).
----------------------------------------
🔄 Agent đang suy nghĩ và dịch sang Cypher...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment) WHERE toLower(p.title) CONTAINS 'overtime' OR toLower(c.description) CONTAINS 'overtime'
RETURN p.title, c.description

Full Context:
[]

> Finished chain.

✨ Agent (18.40s): Chính sách làm thêm giờ tại FPT được quy định như sau:

- Người lao động có thể được yêu cầu làm thêm giờ khi công việc cần thiết và phải được sự đồng ý của người lao động.

- Thời gian làm thêm giờ không quá 2 tiếng trong một ngày, trừ trường hợp đặc biệt do Ban lãnh đạo quyết định.

- Người lao động sẽ được trả lương theo mức quy định tại Điều 96 Bộ luật Lao động năm 2019.

📝 TEST CASE 2/10:
🧑 User: Liệt kê tất cả các đối tượng (Stakeholders) được nhắc đến trong các chính sách của cô

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=17, offset=67>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 67, 'line': 2, 'column': 17}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment)\nWHERE toLower(p.name) CONTAINS 'nghỉ phép'\nRETURN p.title, c.description\n"


Generated Cypher:
MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment)
WHERE toLower(p.name) CONTAINS 'nghỉ phép'
RETURN p.title, c.description

Full Context:
[]

> Finished chain.

✨ Agent (13.37s): FPT có các chính sách sau về nghỉ phép năm:

- Số ngày nghỉ phép năm của nhân viên được tính theo số năm làm việc tại FPT.
- Thời gian nghỉ phép năm phải được thông báo trước ít nhất 3 ngày và không quá 30 ngày so với thời điểm nghỉ phép.
- Trong trường hợp cần thiết, người quản lý có thể yêu cầu nhân viên làm thêm giờ để bù lại số ngày nghỉ phép đã sử dụng.

📝 TEST CASE 6/10:
🧑 User: Những chính sách nào được áp dụng cho Nhân viên (Employee)?
----------------------------------------
🔄 Agent đang suy nghĩ và dịch sang Cypher...


> Entering new GraphCypherQAChain chain...


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `AFFECTS` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=26, offset=25>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 25, 'line': 1, 'column': 26}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (p:PolicyClause)-[:AFFECTS]->(s:Stakeholder) WHERE toLower(s.name) CONTAINS 'nhân viên' RETURN p.title, p.text\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Stakeholder` does not exist in database `neo4

Generated Cypher:
MATCH (p:PolicyClause)-[:AFFECTS]->(s:Stakeholder) WHERE toLower(s.name) CONTAINS 'nhân viên' RETURN p.title, p.text

Full Context:
[]

> Finished chain.

✨ Agent (11.69s): Những chính sách được áp dụng cho nhân viên bao gồm:

- Chính sách lương thưởng
- Chính sách đào tạo và phát triển
- Chính sách phúc lợi
- Chính sách kỷ luật

📝 TEST CASE 7/10:
🧑 User: Chính sách làm thêm giờ (Overtime) bao gồm những quy định chi tiết nào?
----------------------------------------
🔄 Agent đang suy nghĩ và dịch sang Cypher...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment)<-[:HAS_CONSTRAINT]-(con:Constraint)
WHERE toLower(p.title) CONTAINS 'làm thêm giờ' OR toLower(c.description) CONTAINS 'làm thêm giờ'
RETURN p.title, con.value, con.unit, con.period, con.metric

Full Context:
[]

> Finished chain.

✨ Agent (22.37s): Chính sách làm thêm giờ tại FPT được quy định như sau:

- Người lao động được phép làm thêm giờ khi có 

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `AFFECTS` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=22, offset=132>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 132, 'line': 2, 'column': 22}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (p:PolicyClause {title: 'Annual Leave'})-[:CONTAINS]->(c:Commitment)-[:HAS_CONSTRAINT]->(con:Constraint)\nOPTIONAL MATCH (p)-[:AFFECTS]->(s:Stakeholder)\n\nRETURN p.title, con.text, s.name\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn

Generated Cypher:
MATCH (p:PolicyClause {title: 'Annual Leave'})-[:CONTAINS]->(c:Commitment)-[:HAS_CONSTRAINT]->(con:Constraint)
OPTIONAL MATCH (p)-[:AFFECTS]->(s:Stakeholder)

RETURN p.title, con.text, s.name

Full Context:
[]

> Finished chain.

✨ Agent (15.26s): Chính sách Nghỉ phép năm (Annual Leave) của FPT được quy định tại Điều 26, Chương III của Hướng dẫn Thực hiện Quy chế Bổng lộc và Phúc lợi của Công ty Cổ phần Phát triển Công nghệ FPT.

Nghỉ phép năm áp dụng cho tất cả nhân viên FPT.

📝 TEST CASE 9/10:
🧑 User: Mỗi chính sách hiện hành có bao nhiêu quy định (Commitments) đi kèm?
----------------------------------------
🔄 Agent đang suy nghĩ và dịch sang Cypher...


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:PolicyClause)-[:CONTAINS]->(c:Commitment)
RETURN p.title, COUNT(c) AS "Số quy định"


❌ [LỖI TRUY VẤN CYPHER]: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '"Số quy định"': expected a variable name (line 2, column 2